In [1]:
############## Compare the trends of wl and sa measurements over western and central SESE ######
import geopandas as gpd
import geopandas as gpd
from shapely.ops import unary_union, polygonize
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress
from matplotlib.transforms import Affine2D

In [2]:
points = gpd.read_file("/Volumes/MaloneLab/Research/Whitezone/Result/Spatial_Modelling/Results/combo_salinity_points.shp")
# Ensure CRS is geographic (lat/lon), if not reproject to WGS84
if points.crs != "EPSG:4326":
    points = points.to_crs("EPSG:4326")

# Extract lat/lon
points["lon"] = points.geometry.x
points["lat"] = points.geometry.y

# Now you have a DataFrame with lat/lon
print(points[["lat", "lon"]].head())

# 2. Read transects shapefile
transects = gpd.read_file("/Volumes/MaloneLab/Research/Whitezone/Spatial_Data/transects_39_61.shp")

# Reproject if needed to match CRS
if transects.crs != "EPSG:4326":
    transects = transects.to_crs("EPSG:4326")

print(transects.head())

     lat        lon
0  25.22 -80.536581
1  25.14 -80.720000
2  25.18 -80.626919
3  25.25 -80.440000
4  25.23 -80.460000
   ID                                           geometry
0  39  LINESTRING (-80.43141 25.24549, -80.47312 25.3...
1  61  LINESTRING (-80.51986 25.22790, -80.55485 25.2...


In [3]:
####### Read in a csv and convert to shapefile ########
# import pandas as pd
# import geopandas as gpd
# from shapely.geometry import Point

# # Input and output paths
# csv_path = "/Users/bz294/Documents/WhiteZone/Data/Driver_Analysis/Salinity/combo_salinity.csv"
# shp_path = "/Volumes/MaloneLab/Research/Whitezone/Result/Spatial_Modelling/Results/combo_salinity_points.shp"

# # Read CSV
# df = pd.read_csv(csv_path)

# # Convert lat/lon columns to numeric (force errors to NaN, then drop those rows)
# df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
# df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
# df = df.dropna(subset=["lat", "lon"])

# # Create geometry
# geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]

# # Create GeoDataFrame
# gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")  # WGS84

# # Export to shapefile
# gdf.to_file(shp_path, driver="ESRI Shapefile")

# print(f"Shapefile saved at: {shp_path}")


In [4]:
####### Grouping the stations into western and central SESE ##############
# Define station groups
west = [
    "FLAB12_salinity", 
    "FLAB11_salinity", 
    "FLAB10_salinity"
]

central = [
    "FLAB08_salinity", 
    "FLAB07_salinity", 
]

east = [
     #"BISC00B_salinity", 
     #"BISC06B_salinity",
     #"BISC04B_salinity",
     #"BISCA2_salinity", 
     #"BISCA4_salinity"
    "FLAB04_salinity"
]

In [5]:
### EDEN data org ######
salinity_df = pd.read_csv('/Volumes/MaloneLab/Research/Whitezone/Hydro_Salinity/Salinity/salinity_annual_avg.csv')

# Keep only specified sites
keep_sites = ['FLAB10_salinity', 'FLAB12_salinity', 'FLAB11_salinity', 'FLAB07_salinity', 'FLAB08_salinity', 'FLAB04_salinity','BISC00B_salinity', 'BISC06B_salinity', 'BISC04B_salinity', 'BISCA2_salinity', 'BISCA4_salinity']
salinity_df = salinity_df[salinity_df['site'].isin(keep_sites)]


# Convert types
salinity_df['year'] = salinity_df['year'].astype(int)
salinity_df = salinity_df[salinity_df['year'] != 2025]



In [6]:
#### Plot with average lines #########
%matplotlib agg
plt.rcParams["font.family"] = "Times New Roman"
fs = 16
# Filter each region
west_df = salinity_df[salinity_df["site"].isin(west) & (salinity_df["year"] >= 1991) & (salinity_df["year"] <= 2024)]
central_df = salinity_df[salinity_df["site"].isin(central) & (salinity_df["year"] >= 1991) & (salinity_df["year"] <= 2024)]
east_df = salinity_df[salinity_df["site"].isin(east) & (salinity_df["year"] >= 1991) & (salinity_df["year"] <= 2024)]

# Define regions and colors
regions = {
    "West": (west_df, "#66c2a5"),
    "Central": (central_df, "#fc8d62"),
    "East": (east_df, "#8da0cb")
}

# Store yearly averages
yearly_averages = {}

plt.figure(figsize=(12, 6))

for name, (subdf, color) in regions.items():
    if subdf.empty:
        continue
    
    # Grey lines for individual stations
    for station, sdf in subdf.groupby("site"):
        plt.plot(sdf["year"], sdf["annual_avg"], color="grey", alpha=0.4, linewidth=1, label="_nolegend_")
    
    # Yearly statistics
    grouped = subdf.groupby("year")["annual_avg"]
    mean_vals = grouped.mean()
    min_vals = grouped.min()
    max_vals = grouped.max()
    
    # Store yearly mean for reporting
    yearly_averages[name] = mean_vals
    
    # Shaded region for min-max
    plt.fill_between(mean_vals.index, min_vals, max_vals, color=color, alpha=0.12, edgecolor=color, linewidth=0.5)
    
    # Mean line
    plt.plot(mean_vals.index, mean_vals.values, color=color, linewidth=2, label=name)

# Labels and title
plt.xlabel("Year", fontsize=fs)
plt.ylabel("Salinity (PSU)", fontsize=fs)
plt.title("Annual Mean Salinity", fontsize=fs)

# Legend outside
plt.legend(fontsize=fs, bbox_to_anchor=(1.05, 1), loc="upper left")

# Tick size — now starts from 1991
years = list(range(1991, 2025, 5))  # every 5 years
if years[-1] != 2024:   # ensure 2024 is included
    years.append(2024)

plt.xticks(ticks=years, labels=years, fontsize=fs)
plt.yticks(fontsize=fs)

plt.grid(alpha=0.3)
plt.tight_layout()

# Save figure as TIFF
plt.savefig("/Users/bz294/Documents/WhiteZone/Figure/Backup/sa_plot.tif", dpi=300, bbox_inches="tight")

plt.show()


/var/folders/yw/3l8t5xh50v33m6zjql7k1rzsvhfd38/T/ipykernel_94157/3090242816.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
## De-mean ######

# Filter years 2013–2024
test_df = salinity_df[(salinity_df["year"] >= 2014) & (salinity_df["year"] <= 2024)].copy()

# Subtract station mean instead of 2013 baseline
demeaned_list = []
for station, subdf in test_df.groupby("site"):
    subdf = subdf.copy()
    mean_val = subdf["annual_avg"].mean()
    subdf["demeaned"] = subdf["annual_avg"] - mean_val
    demeaned_list.append(subdf)

demeaned_df = pd.concat(demeaned_list)

# Assign region
def assign_region(site):
    if site in west:
        return "west"
    elif site in central:
        return "central"
    else:
        return "east"

demeaned_df["region"] = demeaned_df["site"].apply(assign_region)

# Calculate slope, p-value, r2 for each region
trends = {}
for region, subdf in demeaned_df.groupby("region"):
    yearly_mean = subdf.groupby("year")["demeaned"].mean().reset_index()
    slope, intercept, r_value, p_value, std_err = linregress(yearly_mean["year"], yearly_mean["demeaned"])
    trends[region] = {"slope": slope, "p_value": p_value, "r2": r_value**2}

trend_df = pd.DataFrame(trends).T
print("Demeaned Salinity Trends (2013–2024):")
print(trend_df)


Demeaned Salinity Trends (2013–2024):
            slope   p_value        r2
central -0.777377  0.167751  0.200084
east    -0.341400  0.195019  0.178839
west    -0.902711  0.072154  0.315484


In [ ]:
# 1. Read the CSV
data = pd.read_csv("/Users/bz294/Downloads/sfwmd-data-202509181246.csv")

# 2. Convert collectDate to datetime (let pandas infer the format, including 2-digit years)
data['collectDate'] = pd.to_datetime(data['collectDate'], infer_datetime_format=True)

# 3. Extract year and calculate annual average
data['year'] = data['collectDate'].dt.year
annual_avg = data.groupby('year')['value'].mean().reset_index()

# 4. Save to CSV
annual_avg.to_csv("/Users/bz294/Downloads/sfwmd-annual-average.csv", index=False)
annual_avg